# CCA Sovereign Models — Unified Pipeline

Runs the three CCA model variants on a single weekly panel and assembles the
joint results panel used in Chapter 2.

- **M0** — Baseline (Gray–Merton structural CCA)
- **M1** — Drift adjustment via oil convenience yield 
- **M2** — Log-normal jump diffusion calibrated to OVX

Each model section saves an intermediate CSV to `../output/final/`; the joint
panel section reads them back and computes DD.


## 0 · Setup

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm

from cca_classes import (
    BaselineCCAPricer,
    ConvenienceYieldCCAPricer,
    LogNormalJumpCCAPricer,
    compute_lcl_usd,
    compute_barrier,
)


In [19]:
# ── Sample ───────────────────────────────────────────────────────────────────
EXPORTERS = ['Saudi Arabia', 'UAE (Abu Dhabi)', 'Qatar', 'Colombia',
             'Mexico', 'Brazil', 'Egypt', 'Malaysia']
CONTROLS  = ['Chile', 'China', 'Indonesia', 'Philippines', 'South Africa',
             'South Korea', 'Thailand', 'Turkey']
STUDY_SOVEREIGNS = EXPORTERS + CONTROLS

# ── Time grid ────────────────────────────────────────────
T          = 5.0
FREQ       = 'W'
VOL_WINDOW = 52
ANN_FACTOR = np.sqrt(52)

# ── Estimation window ────────────────────────────────────────────────────────
START_DATE = '2015-01-01'
END_DATE   = '2024-12-31'

# ── Model parameters ─────────────────────────────────────────────────────────
GAMMA   =  0.25            # dampening (shared by M1 and M2)
MU_J    = -0.0116          # M2: mean log-jump size  (Chapter_01/01_Jump_estimation)
SIGMA_J =  0.0996          # M2: std  log-jump size  (Chapter_01/01_Jump_estimation)

# Sigmoid mapping OVX → annualised jump intensity λ
LAMBDA_L  = 0.2533
LAMBDA_K  = 0.0341
LAMBDA_X0 = 40.0

# Futures horizon used to back out the convenience yield (12-month)
T_FUT = 1.0

# ── Paths ────────────────────────────────────────────────────────────────────
PANEL_PATH    = '../data/CCA_panel.csv'
OIL_PRICES    = '../data/oil_prices_datastream.csv'
OIL_FUTURES   = '../data/oil_futures.csv'
OVX_PATH      = '../data/OVXCLS.csv'
OUTPUT_DIR    = '../output/final'

DAMP_TAG   = f"{int(round(GAMMA * 100)):02d}_dampening"
PATH_M0    = f"{OUTPUT_DIR}/M0_results_5YCDS_weekly.csv"
PATH_M1    = f"{OUTPUT_DIR}/M1_results_5YCDS_weekly_{DAMP_TAG}.csv"
PATH_M2    = f"{OUTPUT_DIR}/M2_results_5YCDS_weekly_{DAMP_TAG}.csv"
PATH_FINAL = f"{OUTPUT_DIR}/complete_results_weekly_{DAMP_TAG}.csv"


## 1 · Load weekly panel & shared blocks

A single weekly panel feeds all three models. `compute_common_blocks` adds
`LCL_usd`, `sigma_lcl`, and `B_f` — used identically in M0, M1, M2.


In [20]:
def load_weekly_panel():
    """Load the country CCA panel and collapse to a strict weekly grid."""
    df = pd.read_csv(PANEL_PATH, parse_dates=['date'])
    df = df[df['country'].isin(STUDY_SOVEREIGNS)].copy()

    df = (
        df.set_index(['date', 'country'])
          .groupby('country')
          .resample(FREQ, level='date')
          .last()
          .reset_index()
          .sort_values(['country', 'date'])
          .reset_index(drop=True)
    )
    return df


def compute_common_blocks(df):
    """Add LCL_usd, sigma_lcl, B_f to a per-country (already weekly) frame."""
    df = df.copy().sort_values('date').reset_index(drop=True)

    df['LCL_usd'] = [
        compute_lcl_usd(m, bd, fx, rd, rf, T)
        for m, bd, fx, rd, rf in zip(
            df['monetary_base_bn_local'],
            df['domestic_debt_bn_local'],
            df['fx_rate'],
            df['domestic_rate'],
            df['risk_free_rate'],
        )
    ]

    log_ret = np.log(df['LCL_usd'] / df['LCL_usd'].shift(1))
    df['sigma_lcl'] = log_ret.rolling(window=VOL_WINDOW).std() * ANN_FACTOR

    df['B_f'] = [
        compute_barrier(debt, rf, T)
        for debt, rf in zip(df['external_debt_bn_usd'], df['risk_free_rate'])
    ]
    return df


panel_weekly = load_weekly_panel()
print(f"Panel: {len(panel_weekly):,} rows · {panel_weekly['country'].nunique()} countries")


Panel: 9,184 rows · 16 countries


## 2 · M0 — Baseline

In [21]:
pricer_m0 = BaselineCCAPricer()
results_m0 = []
print("Starting M0 calibration...")


for country, group in panel_weekly.groupby('country'):
    print(f"  {country}")
    df = compute_common_blocks(group)


    out = {'implied_V': [], 'implied_sigma_V': [], 'cca_converged': []}
    for i, row in df.iterrows():
        cca = pricer_m0.solve_CCA_M0(
            row['LCL_usd'], row['sigma_lcl'], row['B_f'],
            row['risk_free_rate'], T,
        )
        out['implied_V'].append(cca['implied_V'])
        out['implied_sigma_V'].append(cca['implied_sigma_V'])
        out['cca_converged'].append(cca['converged'])

    for col, vals in out.items():
        df[col] = vals
    results_m0.append(df)

results_m0 = pd.concat(results_m0, ignore_index=True)
results_m0 = results_m0[
    (results_m0['date'] >= START_DATE) & (results_m0['date'] <= END_DATE)
].copy()

cols = ['date', 'country', 'cds_spread', 'risk_free_rate',
        'B_f', 'LCL_usd', 'sigma_lcl',
        'implied_V', 'implied_sigma_V', 'cca_converged']
results_m0[cols].to_csv(PATH_M0, index=False)
print(f"M0 saved → {PATH_M0}  ({results_m0['cca_converged'].mean():.1%} converged)")


Starting M0 calibration...
  Brazil
  Chile


KeyboardInterrupt: 

## 3 · M1 — Drift adjustment (oil convenience yield)

Spot Brent and 12-month Brent futures are merged onto the weekly panel via
`merge_asof(direction='nearest')`; the convenience yield is

$$y_t = r^f_t - \frac{1}{T_{fut}} \log\!\left(\frac{F_{t,T}}{S_t}\right).$$


In [22]:
# Spot Brent
oil_prices = pd.read_csv(OIL_PRICES)
oil_prices['date'] = pd.to_datetime(oil_prices['date'], format='%m/%d/%y')
oil_prices = oil_prices[['date', 'Brent']].sort_values('date')

# 12m Brent futures
oil_futures = pd.read_csv(OIL_FUTURES)
oil_futures['date'] = pd.to_datetime(oil_futures['date'], format='%d.%m.%Y')
oil_futures = oil_futures[['date', 'Brent_12m']].sort_values('date')

# merge_asof requires the left frame globally sorted by `on`
panel_m1 = panel_weekly.sort_values('date')
panel_m1 = pd.merge_asof(panel_m1, oil_prices,  on='date', direction='nearest')
panel_m1 = pd.merge_asof(panel_m1, oil_futures, on='date', direction='nearest')

panel_m1['log_basis']         = np.log(panel_m1['Brent_12m'] / panel_m1['Brent'])
panel_m1['convenience_yield'] = panel_m1['risk_free_rate'] - (1.0 / T_FUT) * panel_m1['log_basis']

panel_m1 = panel_m1.sort_values(['country', 'date']).reset_index(drop=True)


In [23]:
pricer_m1 = ConvenienceYieldCCAPricer(gamma=GAMMA)
results_m1 = []
print("Starting M1 calibration...")


for country, group in panel_m1.groupby('country'):
    print(f"  {country}")
    df = compute_common_blocks(group)

    # rolling vol of the convenience yield uses the same window/annualisation
    df['sigma_y'] = (
        df['convenience_yield'].diff().rolling(window=VOL_WINDOW).std() * ANN_FACTOR
    )

    out = {'implied_V': [], 'implied_sigma_V': [], 'cca_converged': []}
    for i, row in df.iterrows():
        cca = pricer_m1.solve_CCA_M1(
            row['LCL_usd'], row['sigma_lcl'], row['B_f'],
            row['risk_free_rate'], row['convenience_yield'], row['sigma_y'], T
        )
        out['implied_V'].append(cca['implied_V'])
        out['implied_sigma_V'].append(cca['implied_sigma_V'])
        out['cca_converged'].append(cca['converged'])

    for col, vals in out.items():
        df[col] = vals
    results_m1.append(df)

results_m1 = pd.concat(results_m1, ignore_index=True)
results_m1 = results_m1[
    (results_m1['date'] >= START_DATE) & (results_m1['date'] <= END_DATE)
].copy()

cols = ['date', 'country', 'cds_spread', 'risk_free_rate',
        'B_f', 'LCL_usd', 'sigma_lcl',
        'convenience_yield',
        'implied_V', 'implied_sigma_V', 'cca_converged']
results_m1[cols].to_csv(PATH_M1, index=False)
print(f"M1 saved → {PATH_M1}  ({results_m1['cca_converged'].mean():.1%} converged)")


Starting M1 calibration...
  Brazil
  Chile
  China
  Colombia
  Egypt
  Indonesia
  Malaysia
  Mexico
  Philippines
  Qatar
  Saudi Arabia
  South Africa
  South Korea
  Thailand
  Turkey
  UAE (Abu Dhabi)
M1 saved → ../output/final/M1_results_5YCDS_weekly_25_dampening.csv  (100.0% converged)


## 4 · M2 — Log-normal jump diffusion

OVX is mapped to an annualised jump intensity via the sigmoid fitted in
`Chapter_01/01_Jump_estimation.ipynb`:

$$\lambda_t = 52 \cdot \frac{L}{1 + \exp(-k\,(\text{OVX}_t - x_0))}.$$

Jump size parameters `MU_J`, `SIGMA_J` are scaled by `GAMMA` exactly as in the
original M2 script.


In [24]:
ovx = pd.read_csv(OVX_PATH)
ovx['date'] = pd.to_datetime(ovx['date'])
ovx = ovx.sort_values('date')
ovx['OVXCLS'] = ovx['OVXCLS'].ffill()

panel_m2 = panel_weekly.sort_values('date')
panel_m2 = pd.merge_asof(panel_m2, ovx[['date', 'OVXCLS']],
                         on='date', direction='backward')
panel_m2 = panel_m2.sort_values(['country', 'date']).reset_index(drop=True)
print(f"OVX NAs after merge: {panel_m2['OVXCLS'].isna().sum()}")


def ovx_to_lambda(ovx_value):
    """Sigmoid: OVX → annualised jump intensity λ."""
    if np.isnan(ovx_value) or ovx_value <= 0:
        return 0.0
    lam_weekly = LAMBDA_L / (1 + np.exp(-LAMBDA_K * (ovx_value - LAMBDA_X0)))
    return lam_weekly * 52


panel_m2['lambda_annual'] = panel_m2['OVXCLS'].apply(ovx_to_lambda)


OVX NAs after merge: 0


In [25]:
pricer_m2 = LogNormalJumpCCAPricer(
    mu_J=MU_J * GAMMA,
    sigma_J=SIGMA_J * GAMMA)
pricer_m2._cache = {}

results_m2 = []
print("Starting M2 calibration...")

for country, group in panel_m2.groupby('country'):
    print(f"  {country}")
    df = compute_common_blocks(group)

    implied_V, implied_sigma_diff, implied_sigma_V, converged = [], [], [], []
    prev = None
    for i, row in df.iterrows():
        cca = pricer_m2.solve_CCA_M2(
            row['LCL_usd'], row['sigma_lcl'], row['B_f'],
            row['risk_free_rate'], T, row['lambda_annual'],
            prev_solution=prev,
        )
        prev = cca if cca['converged'] else prev
        implied_V.append(cca['implied_V'])
        implied_sigma_diff.append(cca['implied_sigma_diff'])
        implied_sigma_V.append(cca['implied_sigma_V'])
        converged.append(cca['converged'])

    df['implied_V']          = implied_V
    df['implied_sigma_diff'] = implied_sigma_diff
    df['implied_sigma_V']    = implied_sigma_V
    df['cca_converged']      = converged
    results_m2.append(df)

results_m2 = pd.concat(results_m2, ignore_index=True)
results_m2 = results_m2[
    (results_m2['date'] >= START_DATE) & (results_m2['date'] <= END_DATE)
].copy()

cols = ['date', 'country', 'cds_spread', 'risk_free_rate',
        'B_f', 'LCL_usd', 'sigma_lcl',
        'lambda_annual',
        'implied_V', 'implied_sigma_V', 'cca_converged']
results_m2[cols].to_csv(PATH_M2, index=False)
print(f"M2 saved → {PATH_M2}  ({results_m2['cca_converged'].mean():.1%} converged)")


Starting M2 calibration...
  Brazil
  Chile
  China
  Colombia
  Egypt
  Indonesia
  Malaysia
  Mexico
  Philippines
  Qatar
  Saudi Arabia
  South Africa
  South Korea
  Thailand
  Turkey
  UAE (Abu Dhabi)
M2 saved → ../output/final/M2_results_5YCDS_weekly_25_dampening.csv  (100.0% converged)


## 5 · Joint results panel — DD and PD

Reads the three CSVs back, merges them on (`date`, `country`), and computes the
distance-to-distress for each model:



In [26]:
PATHS = {'M0': PATH_M0, 'M1': PATH_M1, 'M2': PATH_M2}
dfs = {m: pd.read_csv(p, parse_dates=['date']) for m, p in PATHS.items()}

shared      = ['date', 'country', 'cds_spread', 'risk_free_rate',
               'B_f', 'LCL_usd', 'sigma_lcl']
calibration = ['implied_V', 'implied_sigma_V', 'cca_converged']
extras      = {'M1': ['convenience_yield'], 'M2': ['lambda_annual']}

panel = dfs['M0'][shared + calibration].rename(
    columns={c: f'{c}_M0' for c in calibration}
)

for m in ['M1', 'M2']:
    cols = ['date', 'country'] + calibration + extras[m]
    df = dfs[m][cols].rename(columns={c: f'{c}_{m}' for c in calibration})
    panel = panel.merge(df, on=['date', 'country'], how='inner')

panel = panel[panel['country'].isin(STUDY_SOVEREIGNS)].copy()
panel['group']    = np.where(panel['country'].isin(EXPORTERS), 'Exporter', 'Control')
panel['exporter'] = (panel['group'] == 'Exporter').astype(float)

# DD per model
sqt = np.sqrt(T)
B   = panel['B_f'].values
rf  = panel['risk_free_rate'].values
for m in ['M0', 'M1', 'M2']:
    V  = panel[f'implied_V_{m}'].values
    sV = panel[f'implied_sigma_V_{m}'].values
    panel[f'DD_{m}'] = np.log(V / B) / (sV * sqt)

panel.to_csv(PATH_FINAL, index=False)

print(f"Panel: {len(panel):,} rows · {panel['country'].nunique()} countries")
print(f"Date range: {panel['date'].min().date()} → {panel['date'].max().date()}")
print("Convergence rates:")
for m in ['M0', 'M1', 'M2']:
    print(f"  {m}: {panel[f'cca_converged_{m}'].mean():.1%}")
print(f"Saved → {PATH_FINAL}")


Panel: 8,352 rows · 16 countries
Date range: 2015-01-04 → 2024-12-29
Convergence rates:
  M0: 100.0%
  M1: 100.0%
  M2: 100.0%
Saved → ../output/final/complete_results_weekly_25_dampening.csv
